In [25]:
### Get NEW WSF missing data for 20-band imagery
### Tripoli, Dakar, Garissa
### The retained pixels are then used to calculate
### missing WSF after removing building pixels.


import arcpy
import os
import re
import gc

from arcpy.sa import (
    CellStatistics,
    Con,
    IsNull
)


# ============================================================
# USER INPUTS
# ============================================================

CITY = "garissa"

NUMBER_OF_YEARS = 1


# ============================================================
# PATHS
# ============================================================

WORK_GDB = r"E:\WSF\wsf_work_arc\wsf_work_arc.gdb"

WSF_RASTER = rf"E:\WSF\{CITY}_WSFtracker_20160701-20260101.tif"

BUILDINGS = os.path.join(
    WORK_GDB,
    f"buildings_{CITY}_9_13_cleaned"
)


# ============================================================
# OUTPUT NAMES
# ============================================================

NEW_WSF_RASTER = os.path.join(
    WORK_GDB,
    f"{CITY}_new_WSF_{NUMBER_OF_YEARS}_year"
)

BUILDING_RASTER = os.path.join(
    WORK_GDB,
    f"buildings_{CITY}_9_13_cleaned_PolygonToRaster_{NUMBER_OF_YEARS}year"
)

MISSING_RASTER = os.path.join(
    WORK_GDB,
    f"{CITY}_new_WSF_missing_raster_{NUMBER_OF_YEARS}_year"
)

ALL_POINTS = os.path.join(
    WORK_GDB,
    f"{CITY}_new_WSF_all_points_{NUMBER_OF_YEARS}_year"
)

MISSING_POINTS = os.path.join(
    WORK_GDB,
    f"{CITY}_new_WSF_missing_points_{NUMBER_OF_YEARS}_year"
)

FILTERED_MISSING_POINTS = os.path.join(
    WORK_GDB,
    f"{CITY}_new_WSF_missing_points_filtered_{NUMBER_OF_YEARS}_year"
)


# ============================================================
# FUNCTION TO DELETE EXISTING OUTPUT
# ============================================================

def delete_if_exists(path):

    if arcpy.Exists(path):

        print(
            f"Deleting existing output:\n  {path}"
        )

        try:

            arcpy.management.Delete(path)

        except Exception as e:

            raise RuntimeError(
                f"\nCould not overwrite existing dataset:\n"
                f"{path}\n\n"
                f"It is probably open in ArcGIS Pro.\n"
                f"Remove it from the map/Contents pane "
                f"or close any open attribute tables.\n\n"
                f"Original error:\n{e}"
            )


# ============================================================
# CHECK EXTENSIONS
# ============================================================

arcpy.CheckOutExtension("Spatial")

arcpy.CheckOutExtension("ImageAnalyst")

arcpy.env.overwriteOutput = True


# ============================================================
# 1. READ BAND NAMES
# ============================================================

print("\n========================================")
print("1. FINDING WSF BANDS")
print("========================================")

wsf = arcpy.Raster(
    WSF_RASTER
)

band_names = wsf.bandNames

print(
    f"Total bands: {len(band_names)}"
)


# ------------------------------------------------------------
# Extract years
# ------------------------------------------------------------

band_years = {}

for band_name in band_names:

    match = re.search(
        r"(19|20)\d{2}",
        band_name
    )

    if match:

        year = int(
            match.group()
        )

        if year not in band_years:

            band_years[year] = []

        band_years[year].append(
            band_name
        )


if not band_years:

    raise RuntimeError(
        "No four-digit years could be found "
        "in the WSF band names."
    )


available_years = sorted(
    band_years.keys()
)


print(
    "\nAvailable years:"
)

print(
    available_years
)


# ============================================================
# 2. DETERMINE RECENT AND HISTORICAL YEARS
# ============================================================

if NUMBER_OF_YEARS >= len(available_years):

    raise ValueError(
        f"NUMBER_OF_YEARS={NUMBER_OF_YEARS} leaves "
        f"no historical period. There must be at least "
        f"one year before the selected period."
    )


selected_years = available_years[
    -NUMBER_OF_YEARS:
]

historical_years = available_years[
    :-NUMBER_OF_YEARS
]


print(
    "\n========================================"
)

print(
    "RECENT PERIOD"
)

print(
    "========================================"
)

print(
    selected_years
)


print(
    "\n========================================"
)

print(
    "HISTORICAL PERIOD"
)

print(
    "========================================"
)

print(
    historical_years
)


# ============================================================
# 3. GET BAND LISTS
# ============================================================

recent_bands = []

for year in selected_years:

    recent_bands.extend(
        band_years[year]
    )


historical_bands = []

for year in historical_years:

    historical_bands.extend(
        band_years[year]
    )


print(
    "\nRecent bands:"
)

for band in recent_bands:

    print(
        f"  {band}"
    )


print(
    "\nHistorical bands:"
)

for band in historical_bands:

    print(
        f"  {band}"
    )


# ============================================================
# 4. CALCULATE MAXIMUM WSF DURING RECENT PERIOD
# ============================================================

print(
    "\n========================================"
)

print(
    "4. CALCULATING RECENT WSF"
)

print(
    "========================================"
)


recent_rasters = []

for band_name in recent_bands:

    print(
        f"  Adding {band_name}"
    )

    single_band = arcpy.ia.SubsetBands(
        raster=wsf,
        method="BY_NAMES",
        bands=band_name,
        missing_band_action="Fail",
        exclude_bad_bands=False
    )

    recent_rasters.append(
        single_band
    )


recent_max = CellStatistics(
    recent_rasters,
    "MAXIMUM",
    "DATA"
)


# ============================================================
# 5. CALCULATE MAXIMUM WSF DURING HISTORICAL PERIOD
# ============================================================

print(
    "\n========================================"
)

print(
    "5. CALCULATING HISTORICAL WSF"
)

print(
    "========================================"
)


historical_rasters = []

for band_name in historical_bands:

    print(
        f"  Adding {band_name}"
    )

    single_band = arcpy.ia.SubsetBands(
        raster=wsf,
        method="BY_NAMES",
        bands=band_name,
        missing_band_action="Fail",
        exclude_bad_bands=False
    )

    historical_rasters.append(
        single_band
    )


historical_max = CellStatistics(
    historical_rasters,
    "MAXIMUM",
    "DATA"
)


# ============================================================
# 6. IDENTIFY NEW WSF
# ============================================================

print(
    "\n========================================"
)

print(
    "6. IDENTIFYING NEW WSF"
)

print(
    "========================================"
)

print(
    "Keeping pixels where:"
)

print(
    "  Recent WSF > 0"
)

print(
    "  AND"
)

print(
    "  Historical WSF = 0 / NoData"
)


# ------------------------------------------------------------
# IMPORTANT
#
# Do NOT use:
#
#   Con(IsNull(...), None, 1)
#
# The resulting NEW_WSF raster already has NoData
# outside the new-WFS pixels.
# ------------------------------------------------------------

new_wsf = Con(
    (
        (recent_max > 0)
        &
        (
            IsNull(historical_max)
            |
            (historical_max == 0)
        )
    ),
    recent_max
)


# ------------------------------------------------------------
# Delete existing output
# ------------------------------------------------------------

delete_if_exists(
    NEW_WSF_RASTER
)


# ------------------------------------------------------------
# Save new WSF
# ------------------------------------------------------------

new_wsf.save(
    NEW_WSF_RASTER
)


print(
    "\nSaved NEW WSF raster:"
)

print(
    NEW_WSF_RASTER
)


# ------------------------------------------------------------
# Release objects
# ------------------------------------------------------------

new_wsf = None
recent_max = None
historical_max = None
recent_rasters = None
historical_rasters = None
wsf = None

gc.collect()


# ============================================================
# IMPORTANT:
# NEW_WSF_RASTER IS ALREADY THE POSITIVE WSF RASTER
# ============================================================

WSF_POSITIVE_RASTER = NEW_WSF_RASTER


print(
    "\nUsing NEW_WSF_RASTER directly for "
    "RasterToPoint and building filtering."
)

# ============================================================
# 7. RASTERIZE BUILDINGS
# ============================================================

print(
    "\n========================================"
)

print(
    "7. RASTERIZING BUILDINGS"
)

print(
    "========================================"
)


delete_if_exists(
    BUILDING_RASTER
)


# ------------------------------------------------------------
# Get numeric cell size from the WSF raster
# ------------------------------------------------------------

wsf_reference = arcpy.Raster(
    WSF_POSITIVE_RASTER
)

WSF_CELL_SIZE = wsf_reference.meanCellWidth

print(
    f"WSF cell size: {WSF_CELL_SIZE}"
)


# ------------------------------------------------------------
# Rasterize buildings using the WSF grid
#
# snapRaster controls alignment.
# cellSize must be a numeric value.
# ------------------------------------------------------------

with arcpy.EnvManager(
    snapRaster=WSF_POSITIVE_RASTER,
    cellSize=WSF_CELL_SIZE
):

    arcpy.conversion.PolygonToRaster(
        in_features=BUILDINGS,
        value_field="OBJECTID",
        out_rasterdataset=BUILDING_RASTER,
        cell_assignment="CELL_CENTER",
        priority_field="NONE",
        cellsize=WSF_CELL_SIZE,
        build_rat="BUILD"
    )


print(
    "Saved building raster:"
)

print(
    BUILDING_RASTER
)


# ------------------------------------------------------------
# Release reference raster
# ------------------------------------------------------------

wsf_reference = None
gc.collect()


# ============================================================
# 8. REMOVE BUILDING PIXELS
# ============================================================

print(
    "\n========================================"
)

print(
    "8. REMOVING BUILDING PIXELS"
)

print(
    "========================================"
)


delete_if_exists(
    MISSING_RASTER
)


new_wsf_raster = arcpy.Raster(
    WSF_POSITIVE_RASTER
)

building_raster = arcpy.Raster(
    BUILDING_RASTER
)


missing_raster = Con(
    IsNull(
        building_raster
    ),
    new_wsf_raster
)


missing_raster.save(
    MISSING_RASTER
)


print(
    "Saved missing raster:"
)

print(
    MISSING_RASTER
)


missing_raster = None
new_wsf_raster = None
building_raster = None

gc.collect()


# ============================================================
# 9. RASTER TO POINT - ALL NEW WSF
# ============================================================

print(
    "\n========================================"
)

print(
    "9. CREATING ALL NEW WSF POINTS"
)

print(
    "========================================"
)


delete_if_exists(
    ALL_POINTS
)


arcpy.conversion.RasterToPoint(
    in_raster=WSF_POSITIVE_RASTER,
    out_point_features=ALL_POINTS,
    raster_field="Value"
)


print(
    "Saved:"
)

print(
    ALL_POINTS
)


# ============================================================
# 10. RASTER TO POINT - MISSING NEW WSF
# ============================================================

print(
    "\n========================================"
)

print(
    "10. CREATING MISSING NEW WSF POINTS"
)

print(
    "========================================"
)


delete_if_exists(
    MISSING_POINTS
)


arcpy.conversion.RasterToPoint(
    in_raster=MISSING_RASTER,
    out_point_features=MISSING_POINTS,
    raster_field="Value"
)


print(
    "Saved:"
)

print(
    MISSING_POINTS
)


# ============================================================
# 11. REMOVE MISSING POINTS WITHIN 4 METERS
# ============================================================

print(
    "\n========================================"
)

print(
    "11. APPLYING 4-METER BUILDING FILTER"
)

print(
    "========================================"
)


missing_layer = (
    f"{CITY}_new_WSF_missing_"
    f"{NUMBER_OF_YEARS}_year_lyr"
)


if arcpy.Exists(
    missing_layer
):

    try:

        arcpy.management.Delete(
            missing_layer
        )

    except:

        pass


missing_layer = arcpy.management.MakeFeatureLayer(
    MISSING_POINTS,
    missing_layer
).getOutput(0)


print(
    "Selecting missing points more than "
    "4 meters from buildings..."
)


arcpy.management.SelectLayerByLocation(
    in_layer=missing_layer,
    overlap_type="WITHIN_A_DISTANCE",
    select_features=BUILDINGS,
    search_distance="4 Meters",
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="INVERT"
)


selected_count = int(
    arcpy.management.GetCount(
        missing_layer
    )[0]
)


print(
    f"Selected {selected_count:,} "
    f"missing points more than 4 meters "
    f"from buildings."
)


# ============================================================
# 12. SAVE FILTERED MISSING POINTS
# ============================================================

delete_if_exists(
    FILTERED_MISSING_POINTS
)


arcpy.management.CopyFeatures(
    in_features=missing_layer,
    out_feature_class=FILTERED_MISSING_POINTS
)


print(
    "\nSaved filtered missing points:"
)

print(
    FILTERED_MISSING_POINTS
)


# ============================================================
# CLEAN UP
# ============================================================

try:

    arcpy.management.Delete(
        missing_layer
    )

except:

    pass


arcpy.CheckInExtension(
    "Spatial"
)

arcpy.CheckInExtension(
    "ImageAnalyst"
)


# ============================================================
# COMPLETE
# ============================================================

print(
    "\n========================================"
)

print(
    "COMPLETE"
)

print(
    "========================================"
)

print(
    f"City:              {CITY}"
)

print(
    f"Recent years:      {selected_years}"
)

print(
    f"Historical years:  {historical_years}"
)

print(
    "\nLogic:"
)

print(
    "Recent WSF > 0 AND "
    "historical WSF = 0 / NoData"
)

print(
    "\nOutputs:"
)

print(
    f"New WSF:            {NEW_WSF_RASTER}"
)

print(
    f"Building raster:    {BUILDING_RASTER}"
)

print(
    f"Missing raster:     {MISSING_RASTER}"
)

print(
    f"All points:         {ALL_POINTS}"
)

print(
    f"Missing points:     {MISSING_POINTS}"
)

print(
    f"Filtered missing:   {FILTERED_MISSING_POINTS}"
)


1. FINDING WSF BANDS
Total bands: 20

Available years:
[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

RECENT PERIOD
[2026]

HISTORICAL PERIOD
[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Recent bands:
  built_20260101

Historical bands:
  built_20160701
  built_20170101
  built_20170701
  built_20180101
  built_20180701
  built_20190101
  built_20190701
  built_20200101
  built_20200701
  built_20210101
  built_20210701
  built_20220101
  built_20220701
  built_20230101
  built_20230701
  built_20240101
  built_20240701
  built_20250101
  built_20250701

4. CALCULATING RECENT WSF
  Adding built_20260101

5. CALCULATING HISTORICAL WSF
  Adding built_20160701
  Adding built_20170101
  Adding built_20170701
  Adding built_20180101
  Adding built_20180701
  Adding built_20190101
  Adding built_20190701
  Adding built_20200101
  Adding built_20200701
  Adding built_20210101
  Adding built_20210701
  Adding built_20220101
  Adding built_20220701
  Addin

In [ ]:
# Get WSF missing data for 1-band imagery with mode value
#
# Juba, Johannesburg, Kigali
#
# mode interpretation:
#   mode = 1  -> WSF present in all 20 timesteps
#   mode = 20 -> WSF present only in the latest timestep
#
# Therefore:
#   higher mode = newer WSF
#
# Example:
#   20 total timesteps
#   3 years = 6 timesteps
#   minimum mode = 20 - 6 + 1 = 15
#
# So for 3 years, keep mode >= 15.
import os
import gc
import arcpy

from arcpy.sa import (
    Raster,
    SetNull,
    Con,
    IsNull
)


# ============================================================
# 0. INPUTS
# ============================================================

CITY = "kigali"

# Number of most recent years to analyze
NUMBER_OF_YEARS = 3

# WSF has 2 timesteps per year
TIMESTEPS_PER_YEAR = 2

# Total number of WSF timesteps in the input mode raster
TOTAL_TIMESTEPS = 20

# Working geodatabase
WORK_GDB = r"E:\WSF\wsf_work_arc\wsf_work_arc.gdb"

# One-band WSF mode raster
WSF_RASTER = rf"E:\WSF\{CITY}_WSFtracker_20160701-20260101.tif"

# Building footprints
BUILDINGS = os.path.join(
    WORK_GDB,
    f"buildings_{CITY}_9_13_cleaned"
)

# Temporary raster folder
TEMP_DIR = r"E:\WSF\temp_wsf"

os.makedirs(TEMP_DIR, exist_ok=True)

arcpy.env.overwriteOutput = True


# ============================================================
# 1. OUTPUTS
# ============================================================

# ------------------------------------------------------------
# Filtered WSF mode raster
# ------------------------------------------------------------

WSF_MODE_TIF = os.path.join(
    TEMP_DIR,
    f"{CITY}_WSF_mode_{NUMBER_OF_YEARS}_year.tif"
)


# ------------------------------------------------------------
# Rasterized building footprints
# ------------------------------------------------------------

BUILDING_RASTER_TIF = os.path.join(
    TEMP_DIR,
    f"buildings_{CITY}_9_13_cleaned_"
    f"PolygonToRaster_{NUMBER_OF_YEARS}year.tif"
)


# ------------------------------------------------------------
# Missing WSF raster
# ------------------------------------------------------------

MISSING_RASTER_TIF = os.path.join(
    TEMP_DIR,
    f"{CITY}_missing_raster_{NUMBER_OF_YEARS}_year.tif"
)


# ------------------------------------------------------------
# Vector outputs
# ------------------------------------------------------------

ALL_POINTS = os.path.join(
    WORK_GDB,
    f"{CITY}_all_points_{NUMBER_OF_YEARS}_year"
)

MISSING_POINTS = os.path.join(
    WORK_GDB,
    f"{CITY}_missing_points_{NUMBER_OF_YEARS}_year"
)

FILTERED_MISSING_POINTS = os.path.join(
    WORK_GDB,
    f"{CITY}_missing_points_{NUMBER_OF_YEARS}_year_filtered"
)


# ============================================================
# 2. CHECK INPUTS
# ============================================================

print("\n========================================")
print("2. CHECKING INPUTS")
print("========================================")

if not arcpy.Exists(WSF_RASTER):
    raise FileNotFoundError(
        f"WSF raster does not exist:\n{WSF_RASTER}"
    )

if not arcpy.Exists(BUILDINGS):
    raise FileNotFoundError(
        f"Buildings layer does not exist:\n{BUILDINGS}"
    )

print(f"WSF raster: {WSF_RASTER}")
print(f"Buildings:  {BUILDINGS}")


# ============================================================
# 3. FILTER WSF MODE RASTER
# ============================================================

print("\n========================================")
print("3. FILTERING WSF MODE")
print("========================================")

arcpy.CheckOutExtension("Spatial")

# ------------------------------------------------------------
# Read input raster
# ------------------------------------------------------------

wsf_input = Raster(WSF_RASTER)

print(f"Raster bands: {wsf_input.bandCount}")

if wsf_input.bandCount != 1:
    raise ValueError(
        f"This workflow is specifically for the "
        f"one-band WSF mode raster.\n\n"
        f"Input raster has {wsf_input.bandCount} bands."
    )

print("Expected WSF band: mode")


# ------------------------------------------------------------
# Calculate number of timesteps to retain
# ------------------------------------------------------------

REQUIRED_TIMESTEPS = (
    NUMBER_OF_YEARS * TIMESTEPS_PER_YEAR
)

if REQUIRED_TIMESTEPS > TOTAL_TIMESTEPS:
    raise ValueError(
        f"NUMBER_OF_YEARS={NUMBER_OF_YEARS} requires "
        f"{REQUIRED_TIMESTEPS} timesteps, but the raster "
        f"only contains {TOTAL_TIMESTEPS}."
    )


# ------------------------------------------------------------
# Calculate minimum mode to retain
#
# Higher mode = more recent WSF.
#
# 1 year:
#     20 - 2 + 1 = 19
#     retain 19, 20
#
# 2 years:
#     20 - 4 + 1 = 17
#     retain 17-20
#
# 3 years:
#     20 - 6 + 1 = 15
#     retain 15-20
# ------------------------------------------------------------

MIN_MODE = (
    TOTAL_TIMESTEPS
    - REQUIRED_TIMESTEPS
    + 1
)

print(f"Number of years:       {NUMBER_OF_YEARS}")
print(f"Timesteps per year:    {TIMESTEPS_PER_YEAR}")
print(f"Required timesteps:    {REQUIRED_TIMESTEPS}")
print(f"Minimum mode retained: {MIN_MODE}")
print(
    f"Modes retained:        "
    f"{MIN_MODE} through {TOTAL_TIMESTEPS}"
)


# ------------------------------------------------------------
# Delete previous filtered WSF TIFF
# ------------------------------------------------------------

if os.path.exists(WSF_MODE_TIF):

    try:
        os.remove(WSF_MODE_TIF)
        print("\nDeleted previous filtered mode TIFF.")

    except Exception as e:

        raise RuntimeError(
            f"\nCould not delete existing WSF mode TIFF:\n"
            f"{WSF_MODE_TIF}\n\n"
            f"It may be open in ArcGIS Pro.\n\n"
            f"Original error:\n{e}"
        )


# ------------------------------------------------------------
# Filter mode raster
#
# Keep only pixels where:
#
#     mode >= MIN_MODE
#
# All other pixels become NoData.
# ------------------------------------------------------------

mode_raster = Raster(WSF_RASTER)

filtered_mode = SetNull(
    mode_raster < MIN_MODE,
    mode_raster
)


# ------------------------------------------------------------
# Save filtered WSF as TIFF
# ------------------------------------------------------------

filtered_mode.save(
    WSF_MODE_TIF
)

print("\nSaved filtered WSF mode raster:")
print(WSF_MODE_TIF)


# ------------------------------------------------------------
# Release raster objects
# ------------------------------------------------------------

filtered_mode = None
mode_raster = None
wsf_input = None

gc.collect()


# ============================================================
# 4. VERIFY FILTERED WSF
# ============================================================

print("\n========================================")
print("4. VERIFYING FILTERED WSF")
print("========================================")

wsf_reference = Raster(
    WSF_MODE_TIF
)

WSF_CELL_SIZE = wsf_reference.meanCellWidth

print(f"Cell size: {WSF_CELL_SIZE}")
print(f"Rows:      {wsf_reference.height}")
print(f"Columns:   {wsf_reference.width}")
print(f"Bands:     {wsf_reference.bandCount}")

print(
    f"\nUsing WSF cell size: "
    f"{WSF_CELL_SIZE}"
)


# ============================================================
# 5. RASTERIZE BUILDINGS
# ============================================================

print("\n========================================")
print("5. RASTERIZING BUILDINGS")
print("========================================")


# ------------------------------------------------------------
# Delete previous building raster TIFF
# ------------------------------------------------------------

if os.path.exists(BUILDING_RASTER_TIF):

    try:
        os.remove(BUILDING_RASTER_TIF)
        print("Deleted previous building raster TIFF.")

    except Exception as e:

        raise RuntimeError(
            f"\nCould not delete existing building raster:\n"
            f"{BUILDING_RASTER_TIF}\n\n"
            f"It may be open in ArcGIS Pro.\n\n"
            f"Original error:\n{e}"
        )


# ------------------------------------------------------------
# Rasterize buildings
#
# Use:
#   - WSF as snap raster
#   - WSF numeric cell size
#   - OBJECTID as value
# ------------------------------------------------------------

with arcpy.EnvManager(
    snapRaster=WSF_MODE_TIF,
    cellSize=WSF_CELL_SIZE
):

    arcpy.conversion.PolygonToRaster(
        in_features=BUILDINGS,
        value_field="OBJECTID",
        out_rasterdataset=BUILDING_RASTER_TIF,
        cell_assignment="CELL_CENTER",
        priority_field="NONE",
        cellsize=WSF_CELL_SIZE,
        build_rat="BUILD"
    )


print("\nSaved building raster:")
print(BUILDING_RASTER_TIF)


# ------------------------------------------------------------
# Release reference raster
# ------------------------------------------------------------

wsf_reference = None

gc.collect()


# ============================================================
# 6. REMOVE BUILDING PIXELS
# ============================================================

print("\n========================================")
print("6. REMOVING BUILDING PIXELS")
print("========================================")


# ------------------------------------------------------------
# Delete previous missing raster TIFF
# ------------------------------------------------------------

if os.path.exists(MISSING_RASTER_TIF):

    try:
        os.remove(MISSING_RASTER_TIF)
        print("Deleted previous missing raster TIFF.")

    except Exception as e:

        raise RuntimeError(
            f"\nCould not delete existing missing raster:\n"
            f"{MISSING_RASTER_TIF}\n\n"
            f"It may be open in ArcGIS Pro.\n\n"
            f"Original error:\n{e}"
        )


# ------------------------------------------------------------
# Load rasters
# ------------------------------------------------------------

wsf_positive = Raster(
    WSF_MODE_TIF
)

building_raster = Raster(
    BUILDING_RASTER_TIF
)


# ------------------------------------------------------------
# Keep WSF pixels where there is NO building.
#
# The filtered mode raster already contains:
#
#     qualifying WSF = mode value
#     everything else = NoData
#
# Therefore, no additional positive raster is needed.
# ------------------------------------------------------------

output_raster = Con(
    IsNull(building_raster),
    wsf_positive
)


# ------------------------------------------------------------
# Save as TIFF
#
# This avoids ERROR 010240 / FGDBR.
# ------------------------------------------------------------

output_raster.save(
    MISSING_RASTER_TIF
)

print("\nSaved missing WSF raster:")
print(MISSING_RASTER_TIF)


# ------------------------------------------------------------
# Release raster objects
# ------------------------------------------------------------

output_raster = None
wsf_positive = None
building_raster = None

gc.collect()


# ============================================================
# 7. CONVERT ALL FILTERED WSF TO POINTS
# ============================================================

print("\n========================================")
print("7. CONVERTING WSF TO POINTS")
print("========================================")


# ------------------------------------------------------------
# Delete existing all-points output
# ------------------------------------------------------------

if arcpy.Exists(ALL_POINTS):

    try:
        arcpy.management.Delete(
            ALL_POINTS
        )

        print("Deleted previous all-points output.")

    except Exception as e:

        raise RuntimeError(
            f"\nCould not delete existing all-points output:\n"
            f"{ALL_POINTS}\n\n"
            f"Original error:\n{e}"
        )


# ------------------------------------------------------------
# Delete existing missing-points output
# ------------------------------------------------------------

if arcpy.Exists(MISSING_POINTS):

    try:
        arcpy.management.Delete(
            MISSING_POINTS
        )

        print("Deleted previous missing-points output.")

    except Exception as e:

        raise RuntimeError(
            f"\nCould not delete existing missing-points output:\n"
            f"{MISSING_POINTS}\n\n"
            f"Original error:\n{e}"
        )


# ------------------------------------------------------------
# Convert all filtered WSF pixels to points
# ------------------------------------------------------------

arcpy.conversion.RasterToPoint(
    in_raster=WSF_MODE_TIF,
    out_point_features=ALL_POINTS,
    raster_field="Value"
)

print("\nSaved all WSF points:")
print(ALL_POINTS)


# ------------------------------------------------------------
# Convert missing WSF pixels to points
# ------------------------------------------------------------

arcpy.conversion.RasterToPoint(
    in_raster=MISSING_RASTER_TIF,
    out_point_features=MISSING_POINTS,
    raster_field="Value"
)

print("\nSaved missing WSF points:")
print(MISSING_POINTS)


# ============================================================
# 8. REMOVE MISSING POINTS WITHIN 4 METERS OF BUILDINGS
# ============================================================

print("\n========================================")
print("8. FILTERING MISSING POINTS")
print("========================================")


# ------------------------------------------------------------
# Delete previous filtered output
# ------------------------------------------------------------

if arcpy.Exists(FILTERED_MISSING_POINTS):

    try:
        arcpy.management.Delete(
            FILTERED_MISSING_POINTS
        )

        print("Deleted previous filtered points.")

    except Exception as e:

        raise RuntimeError(
            f"\nCould not delete existing filtered "
            f"missing points:\n"
            f"{FILTERED_MISSING_POINTS}\n\n"
            f"Original error:\n{e}"
        )


# ------------------------------------------------------------
# Create temporary layer
# ------------------------------------------------------------

missing_layer = "missing_points_layer"

if arcpy.Exists(missing_layer):

    try:
        arcpy.management.Delete(
            missing_layer
        )
    except:
        pass


arcpy.management.MakeFeatureLayer(
    MISSING_POINTS,
    missing_layer
)


# ------------------------------------------------------------
# Select points OUTSIDE 4 meters from buildings
# ------------------------------------------------------------

arcpy.management.SelectLayerByLocation(
    in_layer=missing_layer,
    overlap_type="WITHIN_A_DISTANCE",
    select_features=BUILDINGS,
    search_distance="4 Meters",
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="INVERT"
)


# ------------------------------------------------------------
# Count selected points
# ------------------------------------------------------------

selected_count = int(
    arcpy.management.GetCount(
        missing_layer
    )[0]
)

print(
    f"\nMissing points more than 4 m "
    f"from buildings: {selected_count}"
)


# ------------------------------------------------------------
# Save filtered points
# ------------------------------------------------------------

arcpy.management.CopyFeatures(
    missing_layer,
    FILTERED_MISSING_POINTS
)

print("\nSaved filtered missing points:")
print(FILTERED_MISSING_POINTS)


# ============================================================
# 9. CLEANUP
# ============================================================

print("\n========================================")
print("9. CLEANUP")
print("========================================")


if arcpy.Exists(missing_layer):

    try:
        arcpy.management.Delete(
            missing_layer
        )
    except:
        pass


# Release Python references
gc.collect()


# Check Spatial Analyst back in
try:
    arcpy.CheckInExtension("Spatial")
except:
    pass


# ============================================================
# 10. SUMMARY
# ============================================================

print("\n========================================")
print("COMPLETE")
print("========================================")

print(f"City:                  {CITY}")
print(f"Years selected:        {NUMBER_OF_YEARS}")
print(f"Timesteps per year:    {TIMESTEPS_PER_YEAR}")
print(f"Required timesteps:    {REQUIRED_TIMESTEPS}")
print(f"Minimum mode retained: {MIN_MODE}")

print("\nRaster outputs:")

print("\nFiltered WSF mode:")
print(
    f"  {WSF_MODE_TIF}"
)

print("\nBuilding raster:")
print(
    f"  {BUILDING_RASTER_TIF}"
)

print("\nMissing WSF raster:")
print(
    f"  {MISSING_RASTER_TIF}"
)

print("\nVector outputs:")

print("\nAll WSF points:")
print(
    f"  {ALL_POINTS}"
)

print("\nMissing WSF points:")
print(
    f"  {MISSING_POINTS}"
)

print("\nFiltered missing points:")
print(
    f"  {FILTERED_MISSING_POINTS}"
)

print("\n========================================")

In [ ]:
# Get segment stats for building and WSF pixel counts

# ============================================================
# Calculate segment-level building and WSF variables
#
# Outputs:
#   1. integrated_building_count_2023
#      = count of building points per segment
#
#   2. wsf_pixel_count_2023
#      = count of 2023 WSF points per segment
#
#   3. building_wsf_ratio
#      = building point count / 2023 WSF point count
#
#   4. wsf_pixel_count_2026
#      = count of 2026 WSF points per segment
#
#   5. building_count_estimated_2026
#      = 2026 WSF count * building_wsf_ratio
#
#   6. estimated_missing_count_1_year
#      = missing_1_year * building_wsf_ratio
#
#   7. estimated_missing_count_3_year
#      = missing_3_year * building_wsf_ratio
# ============================================================

import arcpy
import os


# ============================================================
# INPUTS
# ============================================================

SEGMENT_GPKG = r"E:\WSF\wsf_work_arc\wsf_work_arc.gdb"

SEGMENTS = os.path.join(
    SEGMENT_GPKG,
    "segments_6_cities"
)


WORK_GDB2 = r"E:\WSF\wsf_work_arc\wsf_work_arc.gdb"


# ------------------------------------------------------------
# Building points
# ------------------------------------------------------------

BUILDING_POINTS = os.path.join(
    WORK_GDB2,
    "buildings_six_cities_points_2023_older"
)


# ------------------------------------------------------------
# WSF points - 2023
# ------------------------------------------------------------

WSF_POINTS_2023 = os.path.join(
    WORK_GDB2,
    "wsf_points_2023"
)


# ------------------------------------------------------------
# WSF points - 2026
# ------------------------------------------------------------

WSF_POINTS_2026 = os.path.join(
    WORK_GDB2,
    "wsf_pixel_count_2026"
)


# ============================================================
# OUTPUT FIELDS
# ============================================================

BUILDING_COUNT_FIELD = "integrated_building_count_2023"
WSF_COUNT_2023_FIELD = "wsf_pixel_count_2023"
RATIO_FIELD = "building_wsf_ratio"

WSF_COUNT_2026_FIELD = "wsf_pixel_count_2026"
ESTIMATED_BUILDING_2026_FIELD = "building_count_estimated_2026"

ESTIMATED_MISSING_1_YEAR_FIELD = "estimated_missing_count_1_year"
ESTIMATED_MISSING_3_YEAR_FIELD = "estimated_missing_count_3_year"


# ============================================================
# INPUT MISSING COUNT FIELDS
# ============================================================

MISSING_1_YEAR_FIELD = "missing_1_year"
MISSING_3_YEAR_FIELD = "missing_3_year"


# ============================================================
# CHECK INPUTS
# ============================================================

print("Checking inputs...")

for path, name in [
    (SEGMENTS, "Segment layer"),
    (BUILDING_POINTS, "Building points"),
    (WSF_POINTS_2023, "2023 WSF points"),
    (WSF_POINTS_2026, "2026 WSF points")
]:

    if not arcpy.Exists(path):
        raise RuntimeError(
            f"{name} not found:\n{path}"
        )

print("All inputs found.")


# ============================================================
# SEGMENT ID FIELD
# ============================================================

SEGMENT_ID_FIELD = "OBJECTID"

segment_fields = [
    f.name
    for f in arcpy.ListFields(SEGMENTS)
]

if SEGMENT_ID_FIELD not in segment_fields:
    raise RuntimeError(
        f"Field '{SEGMENT_ID_FIELD}' was not found.\n\n"
        f"Available fields:\n"
        + "\n".join(segment_fields)
    )

print(
    f"Using segment ID field: {SEGMENT_ID_FIELD}"
)


# ============================================================
# CHECK MISSING COUNT FIELDS
# ============================================================

for field_name in [
    MISSING_1_YEAR_FIELD,
    MISSING_3_YEAR_FIELD
]:

    if field_name not in segment_fields:
        raise RuntimeError(
            f"Required field '{field_name}' was not found "
            f"in the segment layer.\n\n"
            f"Available fields:\n"
            + "\n".join(segment_fields)
        )

print(
    f"Found required fields: "
    f"{MISSING_1_YEAR_FIELD}, "
    f"{MISSING_3_YEAR_FIELD}"
)


# ============================================================
# ADD OUTPUT FIELDS
# ============================================================

existing_fields = [
    f.name
    for f in arcpy.ListFields(SEGMENTS)
]

for field_name, field_type in [

    (BUILDING_COUNT_FIELD, "LONG"),
    (WSF_COUNT_2023_FIELD, "LONG"),
    (RATIO_FIELD, "DOUBLE"),
    (WSF_COUNT_2026_FIELD, "LONG"),
    (ESTIMATED_BUILDING_2026_FIELD, "DOUBLE"),
    (ESTIMATED_MISSING_1_YEAR_FIELD, "DOUBLE"),
    (ESTIMATED_MISSING_3_YEAR_FIELD, "DOUBLE")

]:

    if field_name not in existing_fields:

        arcpy.management.AddField(
            SEGMENTS,
            field_name,
            field_type
        )

        print(
            f"Added field: {field_name}"
        )


# ============================================================
# TEMPORARY OUTPUTS
# ============================================================

BUILDING_COUNT_JOIN = os.path.join(
    WORK_GDB2,
    "temp_segment_building_counts"
)

WSF_COUNT_2023_JOIN = os.path.join(
    WORK_GDB2,
    "temp_segment_wsf_counts_2023"
)

WSF_COUNT_2026_JOIN = os.path.join(
    WORK_GDB2,
    "temp_segment_wsf_counts_2026"
)


for temp in [
    BUILDING_COUNT_JOIN,
    WSF_COUNT_2023_JOIN,
    WSF_COUNT_2026_JOIN
]:

    if arcpy.Exists(temp):

        print(
            f"Deleting existing temporary output: {temp}"
        )

        arcpy.management.Delete(temp)


# ============================================================
# 1. COUNT BUILDING POINTS
# ============================================================

print("\n============================================")
print("1. COUNTING BUILDING POINTS")
print("============================================")

arcpy.analysis.SpatialJoin(
    target_features=SEGMENTS,
    join_features=BUILDING_POINTS,
    out_feature_class=BUILDING_COUNT_JOIN,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)


building_counts = {}

with arcpy.da.SearchCursor(
    BUILDING_COUNT_JOIN,
    [
        "TARGET_FID",
        "Join_Count"
    ]
) as cursor:

    for target_fid, count in cursor:

        building_counts[target_fid] = count


print(
    f"Counted buildings for "
    f"{len(building_counts):,} segments."
)


# ============================================================
# 2. COUNT 2023 WSF PIXELS
# ============================================================

print("\n============================================")
print("2. COUNTING 2023 WSF PIXELS")
print("============================================")

arcpy.analysis.SpatialJoin(
    target_features=SEGMENTS,
    join_features=WSF_POINTS_2023,
    out_feature_class=WSF_COUNT_2023_JOIN,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)


wsf_counts_2023 = {}

with arcpy.da.SearchCursor(
    WSF_COUNT_2023_JOIN,
    [
        "TARGET_FID",
        "Join_Count"
    ]
) as cursor:

    for target_fid, count in cursor:

        wsf_counts_2023[target_fid] = count


print(
    f"Counted 2023 WSF pixels for "
    f"{len(wsf_counts_2023):,} segments."
)


# ============================================================
# 3. COUNT 2026 WSF PIXELS
# ============================================================

print("\n============================================")
print("3. COUNTING 2026 WSF PIXELS")
print("============================================")

arcpy.analysis.SpatialJoin(
    target_features=SEGMENTS,
    join_features=WSF_POINTS_2026,
    out_feature_class=WSF_COUNT_2026_JOIN,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)


wsf_counts_2026 = {}

with arcpy.da.SearchCursor(
    WSF_COUNT_2026_JOIN,
    [
        "TARGET_FID",
        "Join_Count"
    ]
) as cursor:

    for target_fid, count in cursor:

        wsf_counts_2026[target_fid] = count


print(
    f"Counted 2026 WSF pixels for "
    f"{len(wsf_counts_2026):,} segments."
)


# ============================================================
# 4. UPDATE SEGMENTS
# ============================================================

print("\n============================================")
print("4. UPDATING SEGMENT ATTRIBUTES")
print("============================================")


with arcpy.da.UpdateCursor(
    SEGMENTS,
    [
        SEGMENT_ID_FIELD,
        BUILDING_COUNT_FIELD,
        WSF_COUNT_2023_FIELD,
        RATIO_FIELD,
        WSF_COUNT_2026_FIELD,
        ESTIMATED_BUILDING_2026_FIELD,
        MISSING_1_YEAR_FIELD,
        MISSING_3_YEAR_FIELD,
        ESTIMATED_MISSING_1_YEAR_FIELD,
        ESTIMATED_MISSING_3_YEAR_FIELD
    ]
) as cursor:

    for row in cursor:

        segment_id = row[0]


        # ----------------------------------------------------
        # 2023 building count
        # ----------------------------------------------------

        building_count = building_counts.get(
            segment_id,
            0
        )

        row[1] = building_count


        # ----------------------------------------------------
        # 2023 WSF count
        # ----------------------------------------------------

        wsf_count_2023 = wsf_counts_2023.get(
            segment_id,
            0
        )

        row[2] = wsf_count_2023


        # ----------------------------------------------------
        # Building / WSF ratio
        # ----------------------------------------------------

        if wsf_count_2023 > 0:

            building_wsf_ratio = (
                building_count /
                wsf_count_2023
            )

            row[3] = building_wsf_ratio

        else:

            building_wsf_ratio = None
            row[3] = None


        # ----------------------------------------------------
        # 2026 WSF count
        # ----------------------------------------------------

        wsf_count_2026 = wsf_counts_2026.get(
            segment_id,
            0
        )

        row[4] = wsf_count_2026


        # ----------------------------------------------------
        # Estimated 2026 building count
        #
        # 2026 WSF count * 2023 building/WSF ratio
        # ----------------------------------------------------

        if (
            wsf_count_2026 > 0
            and building_wsf_ratio is not None
        ):

            row[5] = (
                wsf_count_2026 *
                building_wsf_ratio
            )

        else:

            row[5] = None


        # ----------------------------------------------------
        # Get existing missing counts
        # ----------------------------------------------------

        missing_1_year = row[6]
        missing_3_year = row[7]


        # ----------------------------------------------------
        # Estimated missing buildings - 1 year
        #
        # missing_1_year * building_wsf_ratio
        # ----------------------------------------------------

        if (
            missing_1_year is not None
            and building_wsf_ratio is not None
        ):

            row[8] = (
                missing_1_year *
                building_wsf_ratio
            )

        else:

            row[8] = None


        # ----------------------------------------------------
        # Estimated missing buildings - 3 years
        #
        # missing_3_year * building_wsf_ratio
        # ----------------------------------------------------

        if (
            missing_3_year is not None
            and building_wsf_ratio is not None
        ):

            row[9] = (
                missing_3_year *
                building_wsf_ratio
            )

        else:

            row[9] = None


        cursor.updateRow(row)


# ============================================================
# 5. CLEAN UP
# ============================================================

print("\nCleaning up temporary outputs...")

for temp in [
    BUILDING_COUNT_JOIN,
    WSF_COUNT_2023_JOIN,
    WSF_COUNT_2026_JOIN
]:

    if arcpy.Exists(temp):

        arcpy.management.Delete(temp)


# ============================================================
# COMPLETE
# ============================================================

print("\n============================================")
print("DONE")
print("============================================")

print("\nUpdated segment layer:")
print(SEGMENTS)

print("\nCreated/updated attributes:")

print(
    f" - {BUILDING_COUNT_FIELD}"
)

print(
    f" - {WSF_COUNT_2023_FIELD}"
)

print(
    f" - {RATIO_FIELD}"
)

print(
    f" - {WSF_COUNT_2026_FIELD}"
)

print(
    f" - {ESTIMATED_BUILDING_2026_FIELD}"
)

print(
    f" - {ESTIMATED_MISSING_1_YEAR_FIELD}"
)

print(
    f" - {ESTIMATED_MISSING_3_YEAR_FIELD}"
)

print("\nEstimated 2026 building count:")
print(
    "  wsf_pixel_count_2026 * building_wsf_ratio"
)

print("\nEstimated missing building counts:")
print(
    "  estimated_missing_count_1_year = "
    "missing_1_year * building_wsf_ratio"
)

print(
    "  estimated_missing_count_3_year = "
    "missing_3_year * building_wsf_ratio"
)